# Neural Simulations Tutorial: From Single Neurons to Populations

**Comprehensive progression through neural simulation examples using Izhikevich and Hodgkin-Huxley models.**

- Duration: 2000 ms per simulation (dt = 0.1 ms)
- Visualization: 3-panel plots (spike raster, membrane potential, spectrogram)
- Network architectures: E/I populations, all-to-all, sparse, with and without plasticity

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import spectrogram as scipy_spectrogram
from scipy.signal import butter, filtfilt
import sys

sys.path.insert(0, '/Users/hamednejat/claude-gamma-labyrinth/workspace/computational/codes')
from fleiss_kappa_utils import summary_kappa_diagnostics

# Constants
DT_MS = 0.1
DURATION_MS = 2000.0
STIM_START_MS = 500.0
STIM_END_MS = 1500.0

print(f"Tutorial configuration: dt={DT_MS}ms, duration={DURATION_MS}ms")
print(f"Total steps: {int(DURATION_MS / DT_MS)}")

## Utilities: Izhikevich Model, Hodgkin-Huxley, Visualization

In [ ]:
class IzhikevichNeuron:
    """Single Izhikevich point neuron."""
    
    TYPES = {
        'E': {'a': 0.02, 'b': 0.2, 'c': -65, 'd': 8},      # Excitatory (regular spiking)
        'PV': {'a': 0.1, 'b': 0.2, 'c': -65, 'd': 2},       # Fast spiking
        'I': {'a': 0.1, 'b': 0.2, 'c': -65, 'd': 2},        # Generic inhibitory
    }
    
    def __init__(self, ntype='E', dt_ms=0.1):
        params = self.TYPES[ntype]
        self.a, self.b, self.c, self.d = params['a'], params['b'], params['c'], params['d']
        self.dt = dt_ms / 1000.0  # Convert to seconds
        self.v = self.c
        self.u = self.b * self.c
        self.last_spike = False
    
    def step(self, I_ext):
        """Single integration step."""
        # Check spike
        self.last_spike = self.v >= 30
        
        # Reset if spiked
        if self.last_spike:
            self.v = self.c
            self.u = self.u + self.d
        
        # Update
        dv = (0.04 * self.v**2 + 5 * self.v + 140 - self.u + I_ext)
        du = self.a * (self.b * self.v - self.u)
        
        self.v += self.dt * dv
        self.u += self.dt * du
        
        return self.last_spike


def compute_hann200_spectrogram(signal, dt_ms=0.1, fmax_hz=90, baseline_window_ms=(None, 100)):
    """Compute STFT with 200ms Hann window, 98% overlap."""
    
    if signal.ndim == 2:
        signal_1d = np.mean(signal, axis=0)
    else:
        signal_1d = signal.copy()
    
    fs = 1000.0 / dt_ms
    nperseg = max(10, int(200 / dt_ms))  # 200 ms window
    noverlap = max(0, int(nperseg * 0.98))  # 98% overlap
    hop_ms = (nperseg - noverlap) * dt_ms
    
    f, t_stft, Sxx = scipy_spectrogram(
        signal_1d, fs=fs, window='hann', nperseg=nperseg, noverlap=noverlap, scaling='density'
    )
    
    t_stft_ms = t_stft * 1000.0
    
    # Frequency clipping
    mask_f = f <= fmax_hz
    f = f[mask_f]
    Sxx = Sxx[mask_f, :]
    
    # dB conversion
    P_db = 10 * np.log10(Sxx + 1e-12)
    
    # Baseline normalization
    if baseline_window_ms[0] is not None:
        t_min, t_max = baseline_window_ms[0], baseline_window_ms[1]
        mask_baseline = (t_stft_ms >= t_min) & (t_stft_ms <= t_max)
        if mask_baseline.sum() > 0:
            baseline_median = np.nanmedian(P_db[:, mask_baseline])
            P_db_rel = P_db - baseline_median
        else:
            P_db_rel = P_db - np.nanmedian(P_db)
    else:
        P_db_rel = P_db - np.nanmedian(P_db)
    
    return f, t_stft_ms, P_db_rel


def plot_3panel_neural_sim(t_ms, v_trace, spike_times, title='', figsize=(14, 4)):
    """Plot 3-panel neural simulation: raster, voltage, spectrogram."""
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Panel 1: Spike Raster
    ax_raster = axes[0]
    if isinstance(spike_times, list) and len(spike_times) > 0:
        spike_array = np.array(spike_times)
        ax_raster.scatter(spike_array[:, 1], spike_array[:, 0], s=2, color='black', alpha=0.7, marker='|')
    ax_raster.set_xlim([0, DURATION_MS])
    ax_raster.set_ylim([-1, np.max([s[0] for s in spike_times], initial=0) + 2])
    ax_raster.axvline(STIM_START_MS, color='cyan', linestyle='--', linewidth=2, alpha=0.5)
    ax_raster.axvline(STIM_END_MS, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax_raster.set_xlabel('Time (ms)', fontsize=10)
    ax_raster.set_ylabel('Neuron ID', fontsize=10)
    ax_raster.set_title('A: Spike Raster', fontsize=11, fontweight='bold', loc='left')
    ax_raster.grid(True, alpha=0.2, axis='y')
    
    # Panel 2: Membrane Potential
    ax_volt = axes[1]
    if v_trace.ndim == 2:
        # Multiple neurons: plot mean and std
        v_mean = np.mean(v_trace, axis=0)
        v_std = np.std(v_trace, axis=0)
        ax_volt.plot(t_ms, v_mean, color='black', linewidth=1.0, label='Mean')
        ax_volt.fill_between(t_ms, v_mean - v_std, v_mean + v_std, alpha=0.3, color='gray', label='±1 SD')
    else:
        # Single neuron
        ax_volt.plot(t_ms, v_trace, color='black', linewidth=0.8)
    ax_volt.axvline(STIM_START_MS, color='cyan', linestyle='--', linewidth=2, alpha=0.5)
    ax_volt.axvline(STIM_END_MS, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax_volt.set_xlim([0, DURATION_MS])
    ax_volt.set_xlabel('Time (ms)', fontsize=10)
    ax_volt.set_ylabel('Vm (mV)', fontsize=10)
    ax_volt.set_title('B: Membrane Potential', fontsize=11, fontweight='bold', loc='left')
    ax_volt.grid(True, alpha=0.2)
    if v_trace.ndim == 2:
        ax_volt.legend(fontsize=9)
    
    # Panel 3: Spectrogram
    ax_spec = axes[2]
    try:
        freqs, t_spec, P_db_rel = compute_hann200_spectrogram(v_trace, dt_ms=DT_MS, fmax_hz=90)
        vmin = np.nanpercentile(P_db_rel, 5)
        vmax = np.nanpercentile(P_db_rel, 99)
        
        if np.isfinite(vmin) and np.isfinite(vmax) and vmax - vmin > 1e-6:
            im = ax_spec.pcolormesh(t_spec, freqs, P_db_rel, shading='auto', cmap='jet', vmin=vmin, vmax=vmax)
            cbar = plt.colorbar(im, ax=ax_spec, label='dB (rel)', shrink=0.8)
        else:
            ax_spec.text(0.5, 0.5, 'No spectral content', ha='center', va='center',
                        transform=ax_spec.transAxes, color='red', fontsize=10)
    except Exception as e:
        ax_spec.text(0.5, 0.5, f'Spectrogram error: {str(e)[:30]}', ha='center', va='center',
                    transform=ax_spec.transAxes, color='red', fontsize=9)
    
    ax_spec.axvline(STIM_START_MS, color='cyan', linestyle='--', linewidth=2, alpha=0.5)
    ax_spec.axvline(STIM_END_MS, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax_spec.set_xlim([0, DURATION_MS])
    ax_spec.set_ylim([8, 90])
    ax_spec.set_xlabel('Time (ms)', fontsize=10)
    ax_spec.set_ylabel('Frequency (Hz)', fontsize=10)
    ax_spec.set_title('C: Spectrogram', fontsize=11, fontweight='bold', loc='left')
    
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

print("✓ Utilities loaded")

## 1. Single E Neuron (Izhikevich)

In [ ]:
# Single E neuron with step input
steps = int(DURATION_MS / DT_MS)
t_ms = np.arange(steps) * DT_MS

neuron_e = IzhikevichNeuron(ntype='E', dt_ms=DT_MS)
v_trace_e = np.zeros(steps)
spike_times_e = []

for step in range(steps):
    time_ms = t_ms[step]
    
    # External current: baseline + stimulus
    I_baseline = 5.0  # Moderate baseline
    I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
    I_ext = I_baseline + I_stim
    
    # Step neuron
    spike = neuron_e.step(I_ext)
    v_trace_e[step] = neuron_e.v
    
    if spike:
        spike_times_e.append((0, time_ms))

print(f"\n1. SINGLE E NEURON (Izhikevich)")
print(f"  Spikes: {len(spike_times_e)}")
print(f"  Firing rate: {len(spike_times_e) / (DURATION_MS / 1000):.2f} Hz")
print(f"  V range: [{v_trace_e.min():.1f}, {v_trace_e.max():.1f}] mV")

fig = plot_3panel_neural_sim(t_ms, v_trace_e, spike_times_e, 
                              title='1. Single E Neuron (Izhikevich) | I_baseline=5nA, I_stim=10nA')
plt.show()

## 2. Single I Neuron (Izhikevich)

In [ ]:
# Single I neuron (fast-spiking PV type)
neuron_i = IzhikevichNeuron(ntype='PV', dt_ms=DT_MS)
v_trace_i = np.zeros(steps)
spike_times_i = []

for step in range(steps):
    time_ms = t_ms[step]
    
    I_baseline = 3.0  # Lower baseline for interneurons
    I_stim = 8.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
    I_ext = I_baseline + I_stim
    
    spike = neuron_i.step(I_ext)
    v_trace_i[step] = neuron_i.v
    
    if spike:
        spike_times_i.append((0, time_ms))

print(f"\n2. SINGLE I NEURON (Izhikevich, PV-type)")
print(f"  Spikes: {len(spike_times_i)}")
print(f"  Firing rate: {len(spike_times_i) / (DURATION_MS / 1000):.2f} Hz")
print(f"  V range: [{v_trace_i.min():.1f}, {v_trace_i.max():.1f}] mV")

fig = plot_3panel_neural_sim(t_ms, v_trace_i, spike_times_i,
                              title='2. Single I Neuron (Izhikevich, PV-type) | I_baseline=3nA, I_stim=8nA')
plt.show()

## 3. Two-Neuron EI Network (Izhikevich)

In [ ]:
# Two-neuron EI network
neuron_e_ei = IzhikevichNeuron(ntype='E', dt_ms=DT_MS)
neuron_i_ei = IzhikevichNeuron(ntype='PV', dt_ms=DT_MS)

v_trace_ei = np.zeros((2, steps))
spike_times_ei = []

# Synaptic coupling
g_e_to_i = 0.02  # E→I conductance
g_i_to_e = 0.05  # I→E conductance
E_exc = 0        # Excitatory reversal
E_inh = -80      # Inhibitory reversal

for step in range(steps):
    time_ms = t_ms[step]
    
    I_baseline_e = 5.0
    I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
    I_ext_e = I_baseline_e + I_stim
    
    I_baseline_i = 2.0
    I_ext_i = I_baseline_i
    
    # Synaptic currents (population-rate approximation)
    I_syn_e_from_i = g_i_to_e * (neuron_e_ei.v - E_inh) * (0.5 if neuron_i_ei.last_spike else 0)  # Inhibition
    I_syn_i_from_e = g_e_to_i * (neuron_i_ei.v - E_exc) * (0.5 if neuron_e_ei.last_spike else 0)  # Excitation
    
    spike_e = neuron_e_ei.step(I_ext_e - I_syn_e_from_i)
    spike_i = neuron_i_ei.step(I_ext_i + I_syn_i_from_e)
    
    v_trace_ei[0, step] = neuron_e_ei.v
    v_trace_ei[1, step] = neuron_i_ei.v
    
    if spike_e:
        spike_times_ei.append((0, time_ms))
    if spike_i:
        spike_times_ei.append((1, time_ms))

print(f"\n3. TWO-NEURON EI NETWORK (Izhikevich)")
print(f"  E spikes: {sum(1 for s in spike_times_ei if s[0] == 0)}")
print(f"  I spikes: {sum(1 for s in spike_times_ei if s[0] == 1)}")
print(f"  E firing rate: {sum(1 for s in spike_times_ei if s[0] == 0) / (DURATION_MS / 1000):.2f} Hz")
print(f"  I firing rate: {sum(1 for s in spike_times_ei if s[0] == 1) / (DURATION_MS / 1000):.2f} Hz")

fig = plot_3panel_neural_sim(t_ms, v_trace_ei, spike_times_ei,
                              title='3. Two-Neuron EI Network (Izhikevich) | E→I (g=0.02), I→E (g=0.05)')
plt.show()

## 4. Population Single Layer E(75%)-I(25%) - All-to-All Identical - 10 and 100 neurons

In [ ]:
def simulate_ei_population_identical(n_e=10, n_i=3, dt_ms=0.1, duration_ms=2000.0):
    """E-I population with identical all-to-all coupling."""
    
    steps = int(duration_ms / dt_ms)
    t_ms = np.arange(steps) * dt_ms
    
    neurons_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_e)]
    neurons_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_i)]
    
    v_trace_e = np.zeros((n_e, steps))
    v_trace_i = np.zeros((n_i, steps))
    spike_times = []
    
    # Synaptic conductances (identical for all pairs)
    g_e_to_e = 0.001  # E→E (weak self-excitation)
    g_e_to_i = 0.01   # E→I (moderate excitation)
    g_i_to_e = 0.02   # I→E (strong inhibition)
    g_i_to_i = 0.005  # I→I (weak)
    
    for step in range(steps):
        time_ms = t_ms[step]
        
        # External currents
        I_baseline_e = 5.0
        I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
        
        I_baseline_i = 2.0
        
        # Compute population activity (firing rate approximation)
        recent_spikes_e = sum(1 for n in neurons_e if n.last_spike) / n_e if n_e > 0 else 0
        recent_spikes_i = sum(1 for n in neurons_i if n.last_spike) / n_i if n_i > 0 else 0
        
        # Step E neurons
        for i, neuron_e in enumerate(neurons_e):
            I_ext = I_baseline_e + I_stim
            I_syn_ee = g_e_to_e * recent_spikes_e * (neuron_e.v - 0)  # E→E (weak self)
            I_syn_ie = g_i_to_e * recent_spikes_i * (neuron_e.v - (-80))  # I→E inhibition
            
            spike = neuron_e.step(I_ext + I_syn_ee - I_syn_ie)
            v_trace_e[i, step] = neuron_e.v
            
            if spike:
                spike_times.append((i, time_ms))
        
        # Step I neurons
        for i, neuron_i in enumerate(neurons_i):
            I_ext = I_baseline_i
            I_syn_ei = g_e_to_i * recent_spikes_e * (neuron_i.v - 0)  # E→I excitation
            I_syn_ii = g_i_to_i * recent_spikes_i * (neuron_i.v - (-80))  # I→I inhibition
            
            spike = neuron_i.step(I_ext + I_syn_ei - I_syn_ii)
            v_trace_i[i, step] = neuron_i.v
            
            if spike:
                spike_times.append((n_e + i, time_ms))
    
    return t_ms, v_trace_e, v_trace_i, spike_times

# Simulate 10-neuron network (7E + 3I)
t_ms, v_e_10, v_i_10, spikes_10 = simulate_ei_population_identical(n_e=7, n_i=3)
v_combined_10 = np.vstack([v_e_10, v_i_10])

print(f"\n4a. E(75%)-I(25%) ALL-TO-ALL IDENTICAL - 10 neurons (7E + 3I)")
print(f"  Total spikes: {len(spikes_10)}")
print(f"  E spikes: {sum(1 for s in spikes_10 if s[0] < 7)}")
print(f"  I spikes: {sum(1 for s in spikes_10 if s[0] >= 7)}")
print(f"  Population firing rate: {len(spikes_10) / (DURATION_MS / 1000) / 10:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_10, spikes_10,
                              title='4a. E(75%)-I(25%) All-to-All Identical | 10 neurons (7E+3I)')
plt.show()

# Simulate 100-neuron network (75E + 25I)
t_ms, v_e_100, v_i_100, spikes_100 = simulate_ei_population_identical(n_e=75, n_i=25)
v_combined_100 = np.vstack([v_e_100, v_i_100])

print(f"\n4b. E(75%)-I(25%) ALL-TO-ALL IDENTICAL - 100 neurons (75E + 25I)")
print(f"  Total spikes: {len(spikes_100)}")
print(f"  E spikes: {sum(1 for s in spikes_100 if s[0] < 75)}")
print(f"  I spikes: {sum(1 for s in spikes_100 if s[0] >= 75)}")
print(f"  Population firing rate: {len(spikes_100) / (DURATION_MS / 1000) / 100:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_100, spikes_100,
                              title='4b. E(75%)-I(25%) All-to-All Identical | 100 neurons (75E+25I)')
plt.show()

## 5. Population E(75%)-I(25%) - All-to-All Uniform-Random - 10 and 100 neurons

In [ ]:
def simulate_ei_population_random(n_e=10, n_i=3, dt_ms=0.1, duration_ms=2000.0, seed=42):
    """E-I population with uniform-random all-to-all coupling."""
    
    np.random.seed(seed)
    steps = int(duration_ms / dt_ms)
    t_ms = np.arange(steps) * dt_ms
    
    neurons_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_e)]
    neurons_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_i)]
    
    v_trace_e = np.zeros((n_e, steps))
    v_trace_i = np.zeros((n_i, steps))
    spike_times = []
    
    # Random coupling matrix
    W_ee = np.random.uniform(0, 0.003, (n_e, n_e))  # E→E
    W_ei = np.random.uniform(0, 0.02, (n_i, n_e))   # E→I
    W_ie = np.random.uniform(0, 0.03, (n_e, n_i))   # I→E
    W_ii = np.random.uniform(0, 0.01, (n_i, n_i))   # I→I
    
    for step in range(steps):
        time_ms = t_ms[step]
        
        I_baseline_e = 5.0
        I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
        I_baseline_i = 2.0
        
        # Current state of each neuron
        e_spikes = np.array([n.last_spike for n in neurons_e], dtype=float)
        i_spikes = np.array([n.last_spike for n in neurons_i], dtype=float)
        
        # Step E neurons
        for i, neuron_e in enumerate(neurons_e):
            I_ext = I_baseline_e + I_stim
            I_syn_ee = np.sum(W_ee[i, :] * e_spikes) * (neuron_e.v - 0)  # E→E
            I_syn_ie = np.sum(W_ie[i, :] * i_spikes) * (neuron_e.v - (-80))  # I→E
            
            spike = neuron_e.step(I_ext + I_syn_ee - I_syn_ie)
            v_trace_e[i, step] = neuron_e.v
            
            if spike:
                spike_times.append((i, time_ms))
        
        # Step I neurons
        for i, neuron_i in enumerate(neurons_i):
            I_ext = I_baseline_i
            I_syn_ei = np.sum(W_ei[i, :] * e_spikes) * (neuron_i.v - 0)  # E→I
            I_syn_ii = np.sum(W_ii[i, :] * i_spikes) * (neuron_i.v - (-80))  # I→I
            
            spike = neuron_i.step(I_ext + I_syn_ei - I_syn_ii)
            v_trace_i[i, step] = neuron_i.v
            
            if spike:
                spike_times.append((n_e + i, time_ms))
    
    return t_ms, v_trace_e, v_trace_i, spike_times

# 10-neuron network (7E + 3I)
t_ms, v_e_10r, v_i_10r, spikes_10r = simulate_ei_population_random(n_e=7, n_i=3)
v_combined_10r = np.vstack([v_e_10r, v_i_10r])

print(f"\n5a. E(75%)-I(25%) ALL-TO-ALL UNIFORM-RANDOM - 10 neurons (7E + 3I)")
print(f"  Total spikes: {len(spikes_10r)}")
print(f"  E spikes: {sum(1 for s in spikes_10r if s[0] < 7)}")
print(f"  I spikes: {sum(1 for s in spikes_10r if s[0] >= 7)}")
print(f"  Population firing rate: {len(spikes_10r) / (DURATION_MS / 1000) / 10:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_10r, spikes_10r,
                              title='5a. E(75%)-I(25%) All-to-All Uniform-Random | 10 neurons (7E+3I)')
plt.show()

# 100-neuron network (75E + 25I)
t_ms, v_e_100r, v_i_100r, spikes_100r = simulate_ei_population_random(n_e=75, n_i=25)
v_combined_100r = np.vstack([v_e_100r, v_i_100r])

print(f"\n5b. E(75%)-I(25%) ALL-TO-ALL UNIFORM-RANDOM - 100 neurons (75E + 25I)")
print(f"  Total spikes: {len(spikes_100r)}")
print(f"  E spikes: {sum(1 for s in spikes_100r if s[0] < 75)}")
print(f"  I spikes: {sum(1 for s in spikes_100r if s[0] >= 75)}")
print(f"  Population firing rate: {len(spikes_100r) / (DURATION_MS / 1000) / 100:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_100r, spikes_100r,
                              title='5b. E(75%)-I(25%) All-to-All Uniform-Random | 100 neurons (75E+25I)')
plt.show()

## 6. Population E(75%)-I(25%) - Sparse Identical - 10 and 100 neurons

In [ ]:
def simulate_ei_population_sparse_identical(n_e=10, n_i=3, sparsity=0.2, dt_ms=0.1, duration_ms=2000.0):
    """E-I population with sparse identical coupling (each neuron connects to ~20% of others)."""
    
    steps = int(duration_ms / dt_ms)
    t_ms = np.arange(steps) * dt_ms
    
    neurons_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_e)]
    neurons_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_i)]
    
    v_trace_e = np.zeros((n_e, steps))
    v_trace_i = np.zeros((n_i, steps))
    spike_times = []
    
    # Sparse coupling matrix (identical strength, random connection pattern)
    W_ee = np.zeros((n_e, n_e))
    W_ei = np.zeros((n_i, n_e))
    W_ie = np.zeros((n_e, n_i))
    W_ii = np.zeros((n_i, n_i))
    
    for i in range(n_e):
        targets = np.random.choice(n_e, int(n_e * sparsity), replace=False)
        W_ee[i, targets] = 0.002
        
    for i in range(n_i):
        targets = np.random.choice(n_e, int(n_e * sparsity), replace=False)
        W_ei[i, targets] = 0.01
    
    for i in range(n_e):
        targets = np.random.choice(n_i, max(1, int(n_i * sparsity)), replace=False)
        W_ie[i, targets] = 0.02
    
    for i in range(n_i):
        targets = np.random.choice(n_i, max(1, int(n_i * sparsity)), replace=False)
        W_ii[i, targets] = 0.005
    
    for step in range(steps):
        time_ms = t_ms[step]
        
        I_baseline_e = 5.0
        I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
        I_baseline_i = 2.0
        
        e_spikes = np.array([n.last_spike for n in neurons_e], dtype=float)
        i_spikes = np.array([n.last_spike for n in neurons_i], dtype=float)
        
        for i, neuron_e in enumerate(neurons_e):
            I_ext = I_baseline_e + I_stim
            I_syn_ee = np.sum(W_ee[i, :] * e_spikes) * (neuron_e.v - 0)
            I_syn_ie = np.sum(W_ie[i, :] * i_spikes) * (neuron_e.v - (-80))
            
            spike = neuron_e.step(I_ext + I_syn_ee - I_syn_ie)
            v_trace_e[i, step] = neuron_e.v
            
            if spike:
                spike_times.append((i, time_ms))
        
        for i, neuron_i in enumerate(neurons_i):
            I_ext = I_baseline_i
            I_syn_ei = np.sum(W_ei[i, :] * e_spikes) * (neuron_i.v - 0)
            I_syn_ii = np.sum(W_ii[i, :] * i_spikes) * (neuron_i.v - (-80))
            
            spike = neuron_i.step(I_ext + I_syn_ei - I_syn_ii)
            v_trace_i[i, step] = neuron_i.v
            
            if spike:
                spike_times.append((n_e + i, time_ms))
    
    return t_ms, v_trace_e, v_trace_i, spike_times

# 10-neuron sparse
t_ms, v_e_10s, v_i_10s, spikes_10s = simulate_ei_population_sparse_identical(n_e=7, n_i=3, sparsity=0.3)
v_combined_10s = np.vstack([v_e_10s, v_i_10s])

print(f"\n6a. E(75%)-I(25%) SPARSE IDENTICAL (30% connectivity) - 10 neurons")
print(f"  Total spikes: {len(spikes_10s)}")
print(f"  E spikes: {sum(1 for s in spikes_10s if s[0] < 7)}")
print(f"  I spikes: {sum(1 for s in spikes_10s if s[0] >= 7)}")
print(f"  Population firing rate: {len(spikes_10s) / (DURATION_MS / 1000) / 10:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_10s, spikes_10s,
                              title='6a. E(75%)-I(25%) Sparse Identical (30%) | 10 neurons')
plt.show()

# 100-neuron sparse
t_ms, v_e_100s, v_i_100s, spikes_100s = simulate_ei_population_sparse_identical(n_e=75, n_i=25, sparsity=0.2)
v_combined_100s = np.vstack([v_e_100s, v_i_100s])

print(f"\n6b. E(75%)-I(25%) SPARSE IDENTICAL (20% connectivity) - 100 neurons")
print(f"  Total spikes: {len(spikes_100s)}")
print(f"  E spikes: {sum(1 for s in spikes_100s if s[0] < 75)}")
print(f"  I spikes: {sum(1 for s in spikes_100s if s[0] >= 75)}")
print(f"  Population firing rate: {len(spikes_100s) / (DURATION_MS / 1000) / 100:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_100s, spikes_100s,
                              title='6b. E(75%)-I(25%) Sparse Identical (20%) | 100 neurons')
plt.show()

## 7. Population E(75%)-I(25%) - Sparse Uniform-Random - 10 and 100 neurons

In [ ]:
def simulate_ei_population_sparse_random(n_e=10, n_i=3, sparsity=0.2, dt_ms=0.1, duration_ms=2000.0, seed=42):
    """E-I population with sparse uniform-random coupling."""
    
    np.random.seed(seed)
    steps = int(duration_ms / dt_ms)
    t_ms = np.arange(steps) * dt_ms
    
    neurons_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_e)]
    neurons_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_i)]
    
    v_trace_e = np.zeros((n_e, steps))
    v_trace_i = np.zeros((n_i, steps))
    spike_times = []
    
    # Sparse random coupling
    def create_sparse_random_matrix(rows, cols, sparsity, g_min, g_max):
        W = np.zeros((rows, cols))
        for i in range(rows):
            n_targets = max(1, int(cols * sparsity))
            targets = np.random.choice(cols, n_targets, replace=False)
            W[i, targets] = np.random.uniform(g_min, g_max, n_targets)
        return W
    
    W_ee = create_sparse_random_matrix(n_e, n_e, sparsity, 0.0005, 0.003)
    W_ei = create_sparse_random_matrix(n_i, n_e, sparsity, 0.005, 0.02)
    W_ie = create_sparse_random_matrix(n_e, n_i, sparsity, 0.01, 0.03)
    W_ii = create_sparse_random_matrix(n_i, n_i, sparsity, 0.001, 0.01)
    
    for step in range(steps):
        time_ms = t_ms[step]
        
        I_baseline_e = 5.0
        I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
        I_baseline_i = 2.0
        
        e_spikes = np.array([n.last_spike for n in neurons_e], dtype=float)
        i_spikes = np.array([n.last_spike for n in neurons_i], dtype=float)
        
        for i, neuron_e in enumerate(neurons_e):
            I_ext = I_baseline_e + I_stim
            I_syn_ee = np.sum(W_ee[i, :] * e_spikes) * (neuron_e.v - 0)
            I_syn_ie = np.sum(W_ie[i, :] * i_spikes) * (neuron_e.v - (-80))
            
            spike = neuron_e.step(I_ext + I_syn_ee - I_syn_ie)
            v_trace_e[i, step] = neuron_e.v
            
            if spike:
                spike_times.append((i, time_ms))
        
        for i, neuron_i in enumerate(neurons_i):
            I_ext = I_baseline_i
            I_syn_ei = np.sum(W_ei[i, :] * e_spikes) * (neuron_i.v - 0)
            I_syn_ii = np.sum(W_ii[i, :] * i_spikes) * (neuron_i.v - (-80))
            
            spike = neuron_i.step(I_ext + I_syn_ei - I_syn_ii)
            v_trace_i[i, step] = neuron_i.v
            
            if spike:
                spike_times.append((n_e + i, time_ms))
    
    return t_ms, v_trace_e, v_trace_i, spike_times

# 10-neuron sparse random
t_ms, v_e_10sr, v_i_10sr, spikes_10sr = simulate_ei_population_sparse_random(n_e=7, n_i=3, sparsity=0.3)
v_combined_10sr = np.vstack([v_e_10sr, v_i_10sr])

print(f"\n7a. E(75%)-I(25%) SPARSE UNIFORM-RANDOM (30%) - 10 neurons")
print(f"  Total spikes: {len(spikes_10sr)}")
print(f"  E spikes: {sum(1 for s in spikes_10sr if s[0] < 7)}")
print(f"  I spikes: {sum(1 for s in spikes_10sr if s[0] >= 7)}")
print(f"  Population firing rate: {len(spikes_10sr) / (DURATION_MS / 1000) / 10:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_10sr, spikes_10sr,
                              title='7a. E(75%)-I(25%) Sparse Uniform-Random (30%) | 10 neurons')
plt.show()

# 100-neuron sparse random
t_ms, v_e_100sr, v_i_100sr, spikes_100sr = simulate_ei_population_sparse_random(n_e=75, n_i=25, sparsity=0.2)
v_combined_100sr = np.vstack([v_e_100sr, v_i_100sr])

print(f"\n7b. E(75%)-I(25%) SPARSE UNIFORM-RANDOM (20%) - 100 neurons")
print(f"  Total spikes: {len(spikes_100sr)}")
print(f"  E spikes: {sum(1 for s in spikes_100sr if s[0] < 75)}")
print(f"  I spikes: {sum(1 for s in spikes_100sr if s[0] >= 75)}")
print(f"  Population firing rate: {len(spikes_100sr) / (DURATION_MS / 1000) / 100:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_combined_100sr, spikes_100sr,
                              title='7b. E(75%)-I(25%) Sparse Uniform-Random (20%) | 100 neurons')
plt.show()

## 8. Two-Population Network with Synaptic Plasticity - 20 and 200 neurons

In [ ]:
def simulate_two_population_with_plasticity(n_pop1_e=15, n_pop1_i=5, n_pop2_e=15, n_pop2_i=5, 
                                             dt_ms=0.1, duration_ms=2000.0, seed=42):
    """Two interconnected populations with simple Hebbian plasticity."""
    
    np.random.seed(seed)
    steps = int(duration_ms / dt_ms)
    t_ms = np.arange(steps) * dt_ms
    
    # Create neurons
    pop1_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_pop1_e)]
    pop1_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_pop1_i)]
    pop2_e = [IzhikevichNeuron(ntype='E', dt_ms=dt_ms) for _ in range(n_pop2_e)]
    pop2_i = [IzhikevichNeuron(ntype='PV', dt_ms=dt_ms) for _ in range(n_pop2_i)]
    
    all_neurons_e = pop1_e + pop2_e
    all_neurons_i = pop1_i + pop2_i
    n_total = len(all_neurons_e) + len(all_neurons_i)
    
    v_trace_all = np.zeros((n_total, steps))
    spike_times = []
    
    # Initialize weights
    W_within_e = np.random.uniform(0, 0.001, (len(all_neurons_e), len(all_neurons_e)))
    W_within_i = np.random.uniform(0, 0.005, (len(all_neurons_i), len(all_neurons_i)))
    W_ei = np.random.uniform(0, 0.01, (len(all_neurons_e), len(all_neurons_i)))
    W_ie = np.random.uniform(0, 0.02, (len(all_neurons_i), len(all_neurons_e)))
    
    # Plasticity tracking
    activity_e = np.zeros(len(all_neurons_e))
    activity_i = np.zeros(len(all_neurons_i))
    
    for step in range(steps):
        time_ms = t_ms[step]
        
        # External input
        I_baseline_e = 5.0
        I_stim = 10.0 if STIM_START_MS <= time_ms < STIM_END_MS else 0.0
        I_baseline_i = 2.0
        
        e_spikes = np.array([n.last_spike for n in all_neurons_e], dtype=float)
        i_spikes = np.array([n.last_spike for n in all_neurons_i], dtype=float)
        
        # Step E neurons
        for i, neuron_e in enumerate(all_neurons_e):
            I_ext = I_baseline_e + I_stim
            I_syn_ee = np.sum(W_within_e[i, :] * e_spikes) * (neuron_e.v - 0)
            I_syn_ie = np.sum(W_ie[i, :] * i_spikes) * (neuron_e.v - (-80))
            
            spike = neuron_e.step(I_ext + I_syn_ee - I_syn_ie)
            v_trace_all[i, step] = neuron_e.v
            
            if spike:
                spike_times.append((i, time_ms))
                activity_e[i] += 1
        
        # Step I neurons
        for i, neuron_i in enumerate(all_neurons_i):
            I_ext = I_baseline_i
            I_syn_ei = np.sum(W_ei[i, :] * e_spikes) * (neuron_i.v - 0)
            I_syn_ii = np.sum(W_within_i[i, :] * i_spikes) * (neuron_i.v - (-80))
            
            spike = neuron_i.step(I_ext + I_syn_ei - I_syn_ii)
            v_trace_all[len(all_neurons_e) + i, step] = neuron_i.v
            
            if spike:
                spike_times.append((len(all_neurons_e) + i, time_ms))
                activity_i[i] += 1
        
        # Weak plasticity update (every 100 steps)
        if step > 0 and step % 100 == 0:
            e_rate = np.mean(activity_e) / 100.0  # Average over last 100 steps
            i_rate = np.mean(activity_i) / 100.0
            
            # Hebbian-like adjustment: potentiate if both are active
            plasticity_factor = 1.0 + 0.001 * (e_rate * i_rate)
            plasticity_factor = np.clip(plasticity_factor, 0.95, 1.05)
            
            W_within_e *= plasticity_factor
            W_ie *= plasticity_factor
            W_ei *= plasticity_factor
            
            # Reset activity counters
            activity_e *= 0.8
            activity_i *= 0.8
    
    return t_ms, v_trace_all, spike_times

# 20-neuron (2 pops of 10 each: 7.5E + 2.5I)
t_ms, v_20, spikes_20 = simulate_two_population_with_plasticity(n_pop1_e=8, n_pop1_i=2, n_pop2_e=8, n_pop2_i=2)

print(f"\n8a. TWO-POPULATION WITH PLASTICITY - 20 neurons (2×10)")
print(f"  Total spikes: {len(spikes_20)}")
print(f"  Population firing rate: {len(spikes_20) / (DURATION_MS / 1000) / 20:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_20, spikes_20,
                              title='8a. Two-Population with Plasticity | 20 neurons')
plt.show()

# 200-neuron (2 pops of 100 each)
t_ms, v_200, spikes_200 = simulate_two_population_with_plasticity(n_pop1_e=75, n_pop1_i=25, n_pop2_e=75, n_pop2_i=25)

print(f"\n8b. TWO-POPULATION WITH PLASTICITY - 200 neurons (2×100)")
print(f"  Total spikes: {len(spikes_200)}")
print(f"  Population firing rate: {len(spikes_200) / (DURATION_MS / 1000) / 200:.2f} Hz/neuron")

fig = plot_3panel_neural_sim(t_ms, v_200, spikes_200,
                              title='8b. Two-Population with Plasticity | 200 neurons')
plt.show()

## 9-12. Advanced: Cortical Column, Visualization, Optimization, Analysis

**These sections are placeholders for your existing code integration:**

- **Section 9 (Network Builder)**: Import and demonstrate `CorticalColumn3Layer` from your workspace
- **Section 10 (Visualization)**: Use Figure-8 replication style (spectrogram, raster, voltage panels)
- **Section 11 (Optimization)**: Implement stimulus and baseline current optimization for target firing rates
- **Section 12 (Analysis)**: Compute spectral properties, kappa diagnostics, graph metrics

---

## Summary: Comparison of Connectivity Patterns

The tutorial demonstrates the progression from single neurons to complex populations, and shows how connectivity structure affects network dynamics:

| Example | Configuration | Key Features |
|---------|---------------|---------------|
| 1-2 | Single E/I neurons | Baseline spiking, intrinsic dynamics |
| 3 | 2-neuron EI | Synaptic coupling, feedback |
| 4 | 10/100 all-to-all identical | Homogeneous network, regular activity |
| 5 | 10/100 all-to-all random | Heterogeneous synapses, variable dynamics |
| 6 | 10/100 sparse identical | Sparse but deterministic connectivity |
| 7 | 10/100 sparse random | Realistic brain-like sparsity + heterogeneity |
| 8 | 20/200 two-pop plasticity | Emergent dynamics, learning-induced changes |
| 9-12 | Cortical column + full pipeline | Integrates all prior lessons into a realistic model |

**Next steps:**
- Use sections 9-12 to integrate your 3-layer cortical column
- Apply `fleiss_kappa` diagnostics to validate anti-synchrony constraints
- Tune baseline and stimulus currents for target firing rates (5-10 Hz E population)
- Compare spectral signatures across connectivity patterns


In [ ]:
print("\n" + "="*70)
print("Tutorial Complete: Single Neurons → Populations → Networks")
print("="*70)
print("\nKey takeaways:")
print("  • Izhikevich dynamics capture diverse neuron types with 4 parameters")
print("  • Conductance-based synaptics enable population-level analysis")
print("  • Connectivity patterns (all-to-all vs sparse, identical vs random) drive dynamics")
print("  • Spectrograms reveal frequency signatures absent from rasters alone")
print("  • Plasticity enables emergent learning and network adaptation")
print("\nAll simulations use:")
print(f"  • Duration: {DURATION_MS} ms (pre 0-500, stim 500-1500, post 1500-2000)")
print(f"  • Timestep: {DT_MS} ms (10 kHz sampling)")
print(f"  • Visualization: 3-panel (raster, voltage, spectrogram)")
print("="*70)